CNN+LSTM CLASSIFIER MODEL FOR STOCK PRICE DIRECTION USING OHLC

In [61]:
# ==============================
# 1. Imports
# ==============================
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, LSTM,
    Dense, Dropout, BatchNormalization,
    Flatten, Activation, RepeatVector,
    Permute, Multiply, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# ==============================
# REPRODUCIBILITY CONFIGURATION
# ==============================

import os
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
df = pd.read_csv("NIFTY50.csv")

df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)
df = df.sort_index()

df = df[["Open", "High", "Low", "Close"]]

df.dropna(inplace=True)


# ==============================
# 2. Feature Engineering
# ==============================

HORIZON = 5
SEQ_LEN = 30

df["Return"] = np.log(df["Close"] / df["Close"].shift(1))
df["Vol_20"] = df["Return"].rolling(20).std()

# Momentum Features
df["ROC_5"]  = df["Close"].pct_change(5)
df["ROC_10"] = df["Close"].pct_change(10)
df["ROC_20"] = df["Close"].pct_change(20)

df["MA_10"] = df["Close"].rolling(10).mean()
df["MA_20"] = df["Close"].rolling(20).mean()
df["MA_Spread"] = df["MA_10"] - df["MA_20"]

df["HL_Range"] = (df["High"] - df["Low"]) / df["Close"]

# 5-Day Forward Target
df["Future_Return"] = df["Close"].shift(-HORIZON) / df["Close"] - 1
df["Direction"] = (df["Future_Return"] > 0).astype(int)

df.dropna(inplace=True)

# ==============================
# 3. Volatility Filtering
# ==============================

vol_threshold = df["Vol_20"].quantile(0.6)
df = df[df["Vol_20"] > vol_threshold]

# ==============================
# 4. Feature Selection
# ==============================

features = [
    "Return", "Vol_20",
    "ROC_5", "ROC_10", "ROC_20",
    "MA_Spread",
    "HL_Range"
]

# ==============================
# 5. Walk-Forward Validation
# ==============================

results = []

for year in range(2012, 2025):

    train_df = df[df.index.year < year]
    test_df  = df[df.index.year == year]

    if len(test_df) < SEQ_LEN:
        continue

    scaler = StandardScaler()
    scaler.fit(train_df[features])

    train_scaled = scaler.transform(train_df[features])
    test_scaled  = scaler.transform(test_df[features])

    # Create sequences
    def create_sequences(data, labels):
        X, y = [], []
        for i in range(SEQ_LEN, len(data)):
            X.append(data[i-SEQ_LEN:i])
            y.append(labels[i])
        return np.array(X), np.array(y)

    X_train, y_train = create_sequences(train_scaled, train_df["Direction"].values)
    X_test, y_test   = create_sequences(test_scaled, test_df["Direction"].values)

    n_features = X_train.shape[2]

    # ==============================
    # 6. CNN-LSTM + Attention Model
    # ==============================

    input_layer = Input(shape=(SEQ_LEN, n_features))

    x = Conv1D(64, 3, activation="relu")(input_layer)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, activation="relu")(x)
    x = MaxPooling1D(2)(x)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.3)(x)

    # Attention
    attention = Dense(1, activation="tanh")(x)
    attention = Flatten()(attention)
    attention = Activation("softmax")(attention)
    attention = RepeatVector(64)(attention)
    attention = Permute([2,1])(attention)

    x = Multiply()([x, attention])
    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)

    output = Dense(1, activation="sigmoid")(x)

    model = Model(input_layer, output)

    model.compile(
        optimizer=Adam(0.0005),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="AUC")]
    )

    # Class Weights
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = {0: weights[0], 1: weights[1]}

    model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        verbose=0,
        class_weight=class_weights
    )

    # ==============================
    # 7. Predictions
    # ==============================

    y_prob = model.predict(X_test).ravel()

    # Threshold Optimization
    thresholds = np.linspace(0.4, 0.7, 100)
    best_f1 = 0
    best_t = 0.5

    for t in thresholds:
        preds = (y_prob > t).astype(int)
        f1 = f1_score(y_test, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    y_pred = (y_prob > best_t).astype(int)

    results.append({
        "Year": year,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    })

# ==============================
# 8. Final Results
# ==============================

results_df = pd.DataFrame(results)
print("Average Performance:")
print(results_df.mean())

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 404ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 273ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 199ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 275ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step 
Average Performance:
Year        2017.250000
Accuracy       0.543199
F1             0.587045
AUC            0.550344
dtype: float64


In [62]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.56      0.70      0.63        44
           1       0.68      0.54      0.60        52

    accuracy                           0.61        96
   macro avg       0.62      0.62      0.61        96
weighted avg       0.63      0.61      0.61        96

